# Select positive patients and first positive notes

Configure the input paths in the setup cell, then run the cells in order. See the project README for input schemas and model setup.


In [ ]:
from pathlib import Path
import sys

# Locate the project when Jupyter starts in a notebook subfolder.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "smmnlp" / "__init__.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("Start Jupyter inside SMMnlp_v2 or set its working directory there.")
sys.path.insert(0, str(project_root))


In [ ]:
import logging

import pandas as pd

from smmnlp.paths import ProjectPaths
from smmnlp.parsing import parse_diagnosis_label
from smmnlp.preprocessing import add_parsed_labels

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
paths = ProjectPaths.from_environment()
# Override paths here, or set SMMNLP_DATA_ROOT and SMMNLP_MODEL_PATH.

## Patient-level SMM Dx STATUS and DATE 1. Find patients LLM labeled as positive for SMM (n=200, n=83) 2. Identify the first positive SMM encournter 3. Find patients LLM labeled as hx for SMM (n=, n=) 4. Extract the original date of SMM dx


## Read in raw data


In [ ]:
readfile_name = paths.data("Update/Data/Testing/SMMNLP_testing_data.json")
dataset = pd.read_json(readfile_name)[["PatientSSN", "NoteDate", "NoteText", "Note_index"]]

In [ ]:
len(dataset)

In [ ]:
dataset.head()

## Read in already generated results


In [ ]:
readfileresults = paths.data("Random500/Results/testing_fewshot_binary.xlsx")
raw_predictions = pd.read_excel(readfileresults)

In [ ]:
results_s1 = raw_predictions[["PatientSSN", "Pt_index", "Note_index", "SMMDx_LLM_Binary"]]

In [ ]:
results_s1.head()

In [ ]:
readfileresults = paths.data("Second237/Results/phase1_fewshot_binary.xlsx")
raw_predictions = pd.read_excel(readfileresults)

In [ ]:
results_s2 = raw_predictions[["PatientSSN", "Pt_index", "Note_index", "SMMDx_LLM_Binary"]]

In [ ]:
results_s2.head()

In [ ]:
results_all = pd.concat([results_s1, results_s2], ignore_index=True)

In [ ]:
len(results_all)

In [ ]:
results_all.head()

## Combine true and result data for performance


In [ ]:
final = dataset.merge(
    results_all[["PatientSSN", "Note_index", "SMMDx_LLM_Binary"]],
    on=["PatientSSN", "Note_index"],
    how="inner",
)

In [ ]:
len(final)

In [ ]:
final.head(5)

## POST PROCESSING


In [ ]:
llm_columns = ["SMMDx_LLM_Binary"]
final = add_parsed_labels(final, llm_columns, parse_diagnosis_label)

In [ ]:
final.head()

## Identify the first positive SMM encournter


In [ ]:
final_notes_pos = final[final["post_SMMDx_LLM_Binary"] == "Yes"]

In [ ]:
len(final_notes_pos)

In [ ]:
final_notes_pos["NoteDate2"] = pd.to_datetime(final_notes_pos["NoteDate"], format="%d%b%Y")

In [ ]:
earliest_records = final_notes_pos.loc[final_notes_pos.groupby("PatientSSN")["NoteDate2"].idxmin()]

In [ ]:
earliest_records.head(5)

In [ ]:
len(earliest_records)

## Save the earliest note of each LLM positve patients


In [ ]:
savefile_name = paths.data("Update/Data/Testing/EndToEnd_testing_SMMyesdataset.json")
earliest_records.to_json(savefile_name)

In [ ]:
savefile_name = paths.data("Update/Data/Testing/EndToEnd_testing_SMMyesdataset.xlsx")
earliest_records.to_excel(savefile_name, engine="xlsxwriter", sheet_name="Sheet1", index=False)